In [1]:
# Cell 1 - imports & config
import numpy as np
import pandas as pd
import yfinance as yf

# ==========================
# CONFIG
# ==========================
MAG7_MAP = {
    "AAPL.US": "AAPL",
    "MSFT.US": "MSFT",
    "AMZN.US": "AMZN",
    "GOOGL.US": "GOOGL",
    "META.US": "META",
    "TSLA.US": "TSLA",
    "NVDA.US": "NVDA"
}

START_DATE = "2021-01-01"
END_DATE   = "2025-10-31"
OUTPUT_PARQUET_PATH = "/Users/rk8896/DSO585/stock-news-autoencoder/mag7_yf_2021_2025.parquet"

In [2]:
# Cell 2 - download
tickers = list(MAG7_MAP.values())
print(f"Downloading {tickers} from {START_DATE} to {END_DATE} ...")

px = yf.download(
    tickers=tickers,
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=False,   # keep raw Close and Adj Close
    progress=True,
)

# quick guard
if px.empty:
    raise RuntimeError("No data returned from yfinance (DataFrame is empty).")

[*********************100%***********************]  7 of 7 completed


In [3]:
# Cell 3 - tidy to long (one row per date + ticker)
# Select the columns we want (these exist when yfinance returns multiple fields)
fields = ['Open', 'High', 'Low', 'Close', 'Adj Close']

# Defensive: confirm fields exist in px
available_fields = [f for f in fields if f in px.columns.levels[0]]
if set(['Open','High','Low','Close','Adj Close']).difference(available_fields):
    print("Warning: some expected fields not present; available fields:", available_fields)

# Subset to available fields (will error if none)
px_sub = px[available_fields]

# Stack tickers into rows. px_sub has MultiIndex columns (field, ticker)
# After stacking level=1 (ticker), we get index (Date, ticker) and columns = fields
df_long = px_sub.stack(level=1).reset_index()
# rename columns: ['date', 'ticker_yf', <fields...>]
df_long.columns = ['date', 'ticker_yf'] + available_fields

# Map to your original symbol_query naming
rev_map = {v: k for k, v in MAG7_MAP.items()}
df_long['symbol_query'] = df_long['ticker_yf'].map(rev_map)

# Drop unmapped tickers just in case
df_long = df_long.dropna(subset=['symbol_query'])

# Convert date to date only (no timezone)
df_long['date'] = pd.to_datetime(df_long['date']).dt.date

# Sort for downstream groupby
df_long = df_long.sort_values(['symbol_query', 'date']).reset_index(drop=True)

df_long.head()



,date,ticker_yf,Open,High,Low,Close,Adj Close,symbol_query
0,2021-01-04,AAPL,125.974495,129.410004,133.610001,126.760002,133.520004,AAPL.US
1,2021-01-05,AAPL,127.532013,131.009995,131.740005,128.429993,128.889999,AAPL.US
2,2021-01-06,AAPL,123.239059,126.599998,131.050003,126.379997,127.720001,AAPL.US
3,2021-01-07,AAPL,127.444374,130.919998,131.630005,127.860001,128.360001,AAPL.US
4,2021-01-08,AAPL,128.544388,132.050003,132.630005,130.229996,132.429993,AAPL.US


In [4]:
# Cell 4 - derive requested day_* fields
# day_start_value = Open
# day_max_value   = High
# day_min_value   = Low
# day_end_value   = Adj Close (if present) else Close

df = df_long.copy()

# create standardized columns (use numeric values as-is)
df['day_start_value'] = df['Open']
df['day_max_value']   = df['High']
df['day_min_value']   = df['Low']

# prefer adjusted close for end-of-day value if available
if 'Adj Close' in df.columns:
    df['day_end_value'] = df['Adj Close']
    # also keep raw close if useful
    df['day_end_raw_close'] = df['Close']
else:
    df['day_end_value'] = df['Close']
    df['day_end_raw_close'] = df['Close']

# show a few rows
df[['symbol_query', 'date', 'day_start_value', 'day_max_value', 'day_min_value', 'day_end_value', 'day_end_raw_close']].head()

,symbol_query,date,day_start_value,day_max_value,day_min_value,day_end_value,day_end_raw_close
0,AAPL.US,2021-01-04,125.974495,129.410004,133.610001,133.520004,126.760002
1,AAPL.US,2021-01-05,127.532013,131.009995,131.740005,128.889999,128.429993
2,AAPL.US,2021-01-06,123.239059,126.599998,131.050003,127.720001,126.379997
3,AAPL.US,2021-01-07,127.444374,130.919998,131.630005,128.360001,127.860001
4,AAPL.US,2021-01-08,128.544388,132.050003,132.630005,132.429993,130.229996


In [5]:
# Cell 5 - returns using day_end_value
df = df.sort_values(['symbol_query', 'date'])
df['ret_1d'] = df.groupby('symbol_query')['day_end_raw_close'].pct_change(1)
df['ret_log_1d'] = np.log(df['day_end_raw_close'] / df.groupby('symbol_query')['day_end_raw_close'].shift(1))

df['ret_5d'] = df.groupby('symbol_query')['day_end_raw_close'].pct_change(5)
df['ret_log_5d'] = np.log(df['day_end_raw_close'] / df.groupby('symbol_query')['day_end_raw_close'].shift(5))

df['ret_10d'] = df.groupby('symbol_query')['day_end_raw_close'].pct_change(10)
df['ret_log_10d'] = np.log(df['day_end_raw_close'] / df.groupby('symbol_query')['day_end_raw_close'].shift(10))

df['ret_21d'] = df.groupby('symbol_query')['day_end_raw_close'].pct_change(21)
df['ret_log_21d'] = np.log(df['day_end_raw_close'] / df.groupby('symbol_query')['day_end_raw_close'].shift(21))

df[['symbol_query', 'date', 'day_end_value', 'ret_1d', 'ret_log_1d', 'ret_5d', 'ret_log_5d', 
    'ret_10d', 'ret_log_10d', 'ret_21d', 'ret_log_21d']].head()

,symbol_query,date,day_end_value,ret_1d,ret_log_1d,ret_5d,ret_log_5d,ret_10d,ret_log_10d,ret_21d,ret_log_21d
0,AAPL.US,2021-01-04,133.520004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAPL.US,2021-01-05,128.889999,0.013174,0.013088,NaN,NaN,NaN,NaN,NaN,NaN
2,AAPL.US,2021-01-06,127.720001,-0.015962,-0.016091,NaN,NaN,NaN,NaN,NaN,NaN
3,AAPL.US,2021-01-07,128.360001,0.011711,0.011643,NaN,NaN,NaN,NaN,NaN,NaN
4,AAPL.US,2021-01-08,132.429993,0.018536,0.018366,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df.columns

Index(['date', 'ticker_yf', 'Open', 'High', 'Low', 'Close', 'Adj Close',
       'symbol_query', 'day_start_value', 'day_max_value', 'day_min_value',
       'day_end_value', 'day_end_raw_close', 'ret_1d', 'ret_log_1d', 'ret_5d',
       'ret_log_5d', 'ret_10d', 'ret_log_10d', 'ret_21d', 'ret_log_21d'],
      dtype='object')

In [11]:
# Cell 6 - produce a tidy per-symbol-per-day table (one row per symbol/date) - this is already the case
# But if you want to guarantee that for every symbol/date we have a row and fill missing days with NaN/0:
tidy = df[['symbol_query','date','day_start_value','day_max_value','day_min_value','day_end_value','day_end_raw_close','ret_1d','ret_log_1d','ret_5d',
       'ret_log_5d', 'ret_10d', 'ret_log_10d', 'ret_21d', 'ret_log_21d']].copy()

# If you want business-day continuous index with forward/backfill, you can create a complete index.
# Example: ensure continuous trading calendar per ticker (only useful if you want explicit NaN rows for non-trading days)
# Get full date range
full_dates = pd.date_range(start=START_DATE, end=END_DATE, freq='D').date

# Example to create complete grid (symbol x date)
symbols = tidy['symbol_query'].unique()
grid = pd.MultiIndex.from_product([symbols, full_dates], names=['symbol_query','date']).to_frame(index=False)
tidy_full = grid.merge(tidy, on=['symbol_query','date'], how='left')

# tidy_full now has NaNs for non-trading days (you can fill or leave as-is)
tidy_full.head()

,symbol_query,date,day_start_value,day_max_value,day_min_value,day_end_value,day_end_raw_close,ret_1d,ret_log_1d,ret_5d,ret_log_5d,ret_10d,ret_log_10d,ret_21d,ret_log_21d
0,AAPL.US,2021-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAPL.US,2021-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAPL.US,2021-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAPL.US,2021-01-04,125.974495,129.410004,133.610001,133.520004,126.760002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AAPL.US,2021-01-05,127.532013,131.009995,131.740005,128.889999,128.429993,0.013174,0.013088,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# Cell 7 - save the tidy table (you can save 'df' or 'tidy' or 'tidy_full')
output_df = tidy  # pick tidy (one row per observed trading day)
output_df.to_parquet(OUTPUT_PARQUET_PATH, index=False)
print("Saved:", OUTPUT_PARQUET_PATH)

Saved: /Users/rk8896/DSO585/stock-news-autoencoder/mag7_yf_2021_2025.parquet


In [13]:
output_df.head(30)

,symbol_query,date,day_start_value,day_max_value,day_min_value,day_end_value,day_end_raw_close,ret_1d,ret_log_1d,ret_5d,ret_log_5d,ret_10d,ret_log_10d,ret_21d,ret_log_21d
0,AAPL.US,2021-01-04,125.974495,129.410004,133.610001,133.520004,126.760002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAPL.US,2021-01-05,127.532013,131.009995,131.740005,128.889999,128.429993,0.013174,0.013088,NaN,NaN,NaN,NaN,NaN,NaN
2,AAPL.US,2021-01-06,123.239059,126.599998,131.050003,127.720001,126.379997,-0.015962,-0.016091,NaN,NaN,NaN,NaN,NaN,NaN
3,AAPL.US,2021-01-07,127.444374,130.919998,131.630005,128.360001,127.860001,0.011711,0.011643,NaN,NaN,NaN,NaN,NaN,NaN
4,AAPL.US,2021-01-08,128.544388,132.050003,132.630005,132.429993,130.229996,0.018536,0.018366,NaN,NaN,NaN,NaN,NaN,NaN
5,AAPL.US,2021-01-11,125.555862,128.979996,130.169998,129.190002,128.500000,-0.013284,-0.013373,0.013727,0.013633,NaN,NaN,NaN,NaN
6,AAPL.US,2021-01-12,125.380676,128.800003,129.690002,128.500000,126.860001,-0.012763,-0.012845,-0.012224,-0.012300,NaN,NaN,NaN,NaN
7,AAPL.US,2021-01-13,127.415199,130.889999,131.449997,128.759995,128.490005,0.012849,0.012767,0.016696,0.016558,NaN,NaN,NaN,NaN
8,AAPL.US,2021-01-14,125.487755,128.910004,131.000000,130.800003,128.759995,0.002101,0.002099,0.007039,0.007014,NaN,NaN,NaN,NaN
9,AAPL.US,2021-01-15,123.764732,127.139999,130.220001,128.779999,127.000000,-0.013669,-0.013763,-0.024802,-0.025115,NaN,NaN,NaN,NaN


In [14]:
px = yf.download(
    tickers='SPY',
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=False,   # keep raw Close and Adj Close
    progress=True,
)

[*********************100%***********************]  1 of 1 completed


In [15]:
px

Price,Adj Close,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY,SPY
Date,,,,,,
2021-01-04,345.273987,368.790009,375.450012,364.820007,375.309998,110210800
2021-01-05,347.651978,371.329987,372.500000,368.049988,368.100006,66426200
2021-01-06,349.730438,373.549988,376.980011,369.119995,369.709991,107997700
2021-01-07,354.926544,379.100006,379.899994,375.910004,376.100006,68766800
2021-01-08,356.948792,381.260010,381.489990,377.100006,380.589996,71677200
...,...,...,...,...,...,...
2025-10-24,677.250000,677.250000,678.469971,675.650024,676.460022,74356500
2025-10-27,685.239990,685.239990,685.539978,682.119995,682.729980,63339800


In [16]:
px = px.reset_index()

In [17]:
px[['Date','Close']]

Price,Date,Close
Ticker,,SPY
0,2021-01-04,368.790009
1,2021-01-05,371.329987
2,2021-01-06,373.549988
3,2021-01-07,379.100006
4,2021-01-08,381.260010
...,...,...
1208,2025-10-24,677.250000
1209,2025-10-27,685.239990
1210,2025-10-28,687.059998


In [18]:
px.columns = px.columns.get_level_values(0)
px

Price,Date,Adj Close,Close,High,Low,Open,Volume
0,2021-01-04,345.273987,368.790009,375.450012,364.820007,375.309998,110210800
1,2021-01-05,347.651978,371.329987,372.500000,368.049988,368.100006,66426200
2,2021-01-06,349.730438,373.549988,376.980011,369.119995,369.709991,107997700
3,2021-01-07,354.926544,379.100006,379.899994,375.910004,376.100006,68766800
4,2021-01-08,356.948792,381.260010,381.489990,377.100006,380.589996,71677200
...,...,...,...,...,...,...,...
1208,2025-10-24,677.250000,677.250000,678.469971,675.650024,676.460022,74356500
1209,2025-10-27,685.239990,685.239990,685.539978,682.119995,682.729980,63339800
1210,2025-10-28,687.059998,687.059998,688.909973,684.830017,687.049988,61738100
1211,2025-10-29,687.390015,687.390015,689.700012,682.869995,688.719971,85657100


In [19]:
px['symbol_query'] = 'SPY.US'

In [20]:
px

Price,Date,Adj Close,Close,High,Low,Open,Volume,symbol_query
0,2021-01-04,345.273987,368.790009,375.450012,364.820007,375.309998,110210800,SPY.US
1,2021-01-05,347.651978,371.329987,372.500000,368.049988,368.100006,66426200,SPY.US
2,2021-01-06,349.730438,373.549988,376.980011,369.119995,369.709991,107997700,SPY.US
3,2021-01-07,354.926544,379.100006,379.899994,375.910004,376.100006,68766800,SPY.US
4,2021-01-08,356.948792,381.260010,381.489990,377.100006,380.589996,71677200,SPY.US
...,...,...,...,...,...,...,...,...
1208,2025-10-24,677.250000,677.250000,678.469971,675.650024,676.460022,74356500,SPY.US
1209,2025-10-27,685.239990,685.239990,685.539978,682.119995,682.729980,63339800,SPY.US
1210,2025-10-28,687.059998,687.059998,688.909973,684.830017,687.049988,61738100,SPY.US
1211,2025-10-29,687.390015,687.390015,689.700012,682.869995,688.719971,85657100,SPY.US


In [21]:
px = px.rename(columns = {'Date':'date'})

df = px.copy()

# create standardized columns (use numeric values as-is)
df['day_start_value'] = df['Open']
df['day_max_value']   = df['High']
df['day_min_value']   = df['Low']

# prefer adjusted close for end-of-day value if available
if 'Adj Close' in df.columns:
    df['day_end_value'] = df['Adj Close']
    # also keep raw close if useful
    df['day_end_raw_close'] = df['Close']
else:
    df['day_end_value'] = df['Close']
    df['day_end_raw_close'] = df['Close']

# show a few rows
df[['symbol_query', 'date', 'day_start_value', 'day_max_value', 'day_min_value', 'day_end_value', 'day_end_raw_close']].head()

Price,symbol_query,date,day_start_value,day_max_value,day_min_value,day_end_value,day_end_raw_close
0,SPY.US,2021-01-04,375.309998,375.450012,364.820007,345.273987,368.790009
1,SPY.US,2021-01-05,368.100006,372.500000,368.049988,347.651978,371.329987
2,SPY.US,2021-01-06,369.709991,376.980011,369.119995,349.730438,373.549988
3,SPY.US,2021-01-07,376.100006,379.899994,375.910004,354.926544,379.100006
4,SPY.US,2021-01-08,380.589996,381.489990,377.100006,356.948792,381.260010


In [22]:
df = df.sort_values(['symbol_query', 'date'])
df['ret_1d'] = df.groupby('symbol_query')['day_end_raw_close'].pct_change(1)
df['ret_log_1d'] = np.log(df['day_end_raw_close'] / df.groupby('symbol_query')['day_end_raw_close'].shift(1))

df['ret_5d'] = df.groupby('symbol_query')['day_end_raw_close'].pct_change(5)
df['ret_log_5d'] = np.log(df['day_end_raw_close'] / df.groupby('symbol_query')['day_end_raw_close'].shift(5))

df['ret_10d'] = df.groupby('symbol_query')['day_end_raw_close'].pct_change(10)
df['ret_log_10d'] = np.log(df['day_end_raw_close'] / df.groupby('symbol_query')['day_end_raw_close'].shift(10))

df['ret_21d'] = df.groupby('symbol_query')['day_end_raw_close'].pct_change(21)
df['ret_log_21d'] = np.log(df['day_end_raw_close'] / df.groupby('symbol_query')['day_end_raw_close'].shift(21))

df[['symbol_query', 'date', 'day_end_value', 'ret_1d', 'ret_log_1d', 'ret_5d', 'ret_log_5d', 
    'ret_10d', 'ret_log_10d', 'ret_21d', 'ret_log_21d']].head()

Price,symbol_query,date,day_end_value,ret_1d,ret_log_1d,ret_5d,ret_log_5d,ret_10d,ret_log_10d,ret_21d,ret_log_21d
0,SPY.US,2021-01-04,345.273987,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SPY.US,2021-01-05,347.651978,0.006887,0.006864,NaN,NaN,NaN,NaN,NaN,NaN
2,SPY.US,2021-01-06,349.730438,0.005979,0.005961,NaN,NaN,NaN,NaN,NaN,NaN
3,SPY.US,2021-01-07,354.926544,0.014857,0.014748,NaN,NaN,NaN,NaN,NaN,NaN
4,SPY.US,2021-01-08,356.948792,0.005698,0.005682,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
df.tail()

Price,date,Adj Close,Close,High,Low,Open,Volume,symbol_query,day_start_value,day_max_value,...,day_end_value,day_end_raw_close,ret_1d,ret_log_1d,ret_5d,ret_log_5d,ret_10d,ret_log_10d,ret_21d,ret_log_21d
1208,2025-10-24,677.250000,677.250000,678.469971,675.650024,676.460022,74356500,SPY.US,676.460022,678.469971,...,677.250000,677.250000,0.008173,0.008139,0.019356,0.019171,0.037104,0.036433,0.029177,0.028760
1209,2025-10-27,685.239990,685.239990,685.539978,682.119995,682.729980,63339800,SPY.US,682.729980,685.539978,...,685.239990,685.239990,0.011798,0.011729,0.020766,0.020553,0.033482,0.032934,0.035387,0.034776
1210,2025-10-28,687.059998,687.059998,688.909973,684.830017,687.049988,61738100,SPY.US,687.049988,688.909973,...,687.059998,687.059998,0.002656,0.002652,0.023492,0.023220,0.037495,0.036809,0.035228,0.034622
1211,2025-10-29,687.390015,687.390015,689.700012,682.869995,688.719971,85657100,SPY.US,688.719971,689.700012,...,687.390015,687.390015,0.000480,0.000480,0.029335,0.028913,0.033405,0.032859,0.031838,0.031342
1212,2025-10-30,679.830017,679.830017,685.940002,679.830017,683.900024,76335800,SPY.US,683.900024,685.940002,...,679.830017,679.830017,-0.010998,-0.011059,0.012013,0.011942,0.029048,0.028634,0.017024,0.016881


In [24]:
df.columns

Index(['date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume',
       'symbol_query', 'day_start_value', 'day_max_value', 'day_min_value',
       'day_end_value', 'day_end_raw_close', 'ret_1d', 'ret_log_1d', 'ret_5d',
       'ret_log_5d', 'ret_10d', 'ret_log_10d', 'ret_21d', 'ret_log_21d'],
      dtype='object', name='Price')

In [25]:
df = df.rename(columns = {'day_start_value':'market_day_start_value',
                          'day_max_value':'market_day_max_value',
                          'day_min_value':'market_day_min_value',
                          'day_end_value':'market_day_end_value',
                          'day_end_raw_close':'market_day_end_raw_close',
                          'ret_1d':'market_ret_1d',
                          'ret_log_1d':'market_ret_log_1d',
                          'ret_5d':'market_ret_5d',
                          'ret_log_5d':'market_ret_log_5d',
                          'ret_10d':'market_ret_10d',
                          'ret_log_10d':'market_ret_log_10d',
                          'ret_21d':'market_ret_21d',
                          'ret_log_21d':'market_ret_log_21d'
                        })

In [26]:
df.columns

Index(['date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume',
       'symbol_query', 'market_day_start_value', 'market_day_max_value',
       'market_day_min_value', 'market_day_end_value',
       'market_day_end_raw_close', 'market_ret_1d', 'market_ret_log_1d',
       'market_ret_5d', 'market_ret_log_5d', 'market_ret_10d',
       'market_ret_log_10d', 'market_ret_21d', 'market_ret_log_21d'],
      dtype='object', name='Price')

In [27]:
df = df[['date','market_day_start_value', 'market_day_max_value',
       'market_day_min_value', 'market_day_end_value',
       'market_day_end_raw_close', 'market_ret_1d', 'market_ret_log_1d',
       'market_ret_5d', 'market_ret_log_5d', 'market_ret_10d',
       'market_ret_log_10d', 'market_ret_21d', 'market_ret_log_21d']]
df

Price,date,market_day_start_value,market_day_max_value,market_day_min_value,market_day_end_value,market_day_end_raw_close,market_ret_1d,market_ret_log_1d,market_ret_5d,market_ret_log_5d,market_ret_10d,market_ret_log_10d,market_ret_21d,market_ret_log_21d
0,2021-01-04,375.309998,375.450012,364.820007,345.273987,368.790009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-05,368.100006,372.500000,368.049988,347.651978,371.329987,0.006887,0.006864,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-01-06,369.709991,376.980011,369.119995,349.730438,373.549988,0.005979,0.005961,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-01-07,376.100006,379.899994,375.910004,354.926544,379.100006,0.014857,0.014748,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-01-08,380.589996,381.489990,377.100006,356.948792,381.260010,0.005698,0.005682,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1208,2025-10-24,676.460022,678.469971,675.650024,677.250000,677.250000,0.008173,0.008139,0.019356,0.019171,0.037104,0.036433,0.029177,0.028760
1209,2025-10-27,682.729980,685.539978,682.119995,685.239990,685.239990,0.011798,0.011729,0.020766,0.020553,0.033482,0.032934,0.035387,0.034776
1210,2025-10-28,687.049988,688.909973,684.830017,687.059998,687.059998,0.002656,0.002652,0.023492,0.023220,0.037495,0.036809,0.035228,0.034622
1211,2025-10-29,688.719971,689.700012,682.869995,687.390015,687.390015,0.000480,0.000480,0.029335,0.028913,0.033405,0.032859,0.031838,0.031342


In [30]:
df.dtypes

Price
date                        datetime64[ns]
market_day_start_value             float64
market_day_max_value               float64
market_day_min_value               float64
market_day_end_value               float64
market_day_end_raw_close           float64
market_ret_1d                      float64
market_ret_log_1d                  float64
market_ret_5d                      float64
market_ret_log_5d                  float64
market_ret_10d                     float64
market_ret_log_10d                 float64
market_ret_21d                     float64
market_ret_log_21d                 float64
dtype: object

In [34]:
# Access .dt.date as a property
df['date'] = df['date'].dt.date

In [35]:
output_df

,symbol_query,date,day_start_value,day_max_value,day_min_value,day_end_value,day_end_raw_close,ret_1d,ret_log_1d,ret_5d,ret_log_5d,ret_10d,ret_log_10d,ret_21d,ret_log_21d
0,AAPL.US,2021-01-04,125.974495,129.410004,133.610001,133.520004,126.760002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAPL.US,2021-01-05,127.532013,131.009995,131.740005,128.889999,128.429993,0.013174,0.013088,NaN,NaN,NaN,NaN,NaN,NaN
2,AAPL.US,2021-01-06,123.239059,126.599998,131.050003,127.720001,126.379997,-0.015962,-0.016091,NaN,NaN,NaN,NaN,NaN,NaN
3,AAPL.US,2021-01-07,127.444374,130.919998,131.630005,128.360001,127.860001,0.011711,0.011643,NaN,NaN,NaN,NaN,NaN,NaN
4,AAPL.US,2021-01-08,128.544388,132.050003,132.630005,132.429993,130.229996,0.018536,0.018366,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8486,TSLA.US,2025-10-24,433.720001,433.720001,451.679993,446.829987,430.170013,0.039309,0.038556,0.015510,0.015391,0.045498,0.044493,0.026463,0.026119
8487,TSLA.US,2025-10-27,452.420013,452.420013,460.160004,439.980011,438.690002,0.019806,0.019613,-0.004358,-0.004367,0.045247,0.044253,0.041970,0.041113
8488,TSLA.US,2025-10-28,460.549988,460.549988,467.000000,454.779999,451.600006,0.029429,0.029004,0.021604,0.021374,0.080745,0.077650,0.027531,0.027159
8489,TSLA.US,2025-10-29,461.510010,461.510010,465.700012,462.500000,452.649994,0.002325,0.002322,0.055128,0.053662,0.061736,0.059906,0.045091,0.044104


In [36]:
final_df = pd.merge(output_df, df, on = 'date', how='left')

In [37]:
final_df

,symbol_query,date,day_start_value,day_max_value,day_min_value,day_end_value,day_end_raw_close,ret_1d,ret_log_1d,ret_5d,...,market_day_end_value,market_day_end_raw_close,market_ret_1d,market_ret_log_1d,market_ret_5d,market_ret_log_5d,market_ret_10d,market_ret_log_10d,market_ret_21d,market_ret_log_21d
0,AAPL.US,2021-01-04,125.974495,129.410004,133.610001,133.520004,126.760002,NaN,NaN,NaN,...,345.273987,368.790009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAPL.US,2021-01-05,127.532013,131.009995,131.740005,128.889999,128.429993,0.013174,0.013088,NaN,...,347.651978,371.329987,0.006887,0.006864,NaN,NaN,NaN,NaN,NaN,NaN
2,AAPL.US,2021-01-06,123.239059,126.599998,131.050003,127.720001,126.379997,-0.015962,-0.016091,NaN,...,349.730438,373.549988,0.005979,0.005961,NaN,NaN,NaN,NaN,NaN,NaN
3,AAPL.US,2021-01-07,127.444374,130.919998,131.630005,128.360001,127.860001,0.011711,0.011643,NaN,...,354.926544,379.100006,0.014857,0.014748,NaN,NaN,NaN,NaN,NaN,NaN
4,AAPL.US,2021-01-08,128.544388,132.050003,132.630005,132.429993,130.229996,0.018536,0.018366,NaN,...,356.948792,381.260010,0.005698,0.005682,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8486,TSLA.US,2025-10-24,433.720001,433.720001,451.679993,446.829987,430.170013,0.039309,0.038556,0.015510,...,677.250000,677.250000,0.008173,0.008139,0.019356,0.019171,0.037104,0.036433,0.029177,0.028760
8487,TSLA.US,2025-10-27,452.420013,452.420013,460.160004,439.980011,438.690002,0.019806,0.019613,-0.004358,...,685.239990,685.239990,0.011798,0.011729,0.020766,0.020553,0.033482,0.032934,0.035387,0.034776
8488,TSLA.US,2025-10-28,460.549988,460.549988,467.000000,454.779999,451.600006,0.029429,0.029004,0.021604,...,687.059998,687.059998,0.002656,0.002652,0.023492,0.023220,0.037495,0.036809,0.035228,0.034622
8489,TSLA.US,2025-10-29,461.510010,461.510010,465.700012,462.500000,452.649994,0.002325,0.002322,0.055128,...,687.390015,687.390015,0.000480,0.000480,0.029335,0.028913,0.033405,0.032859,0.031838,0.031342


In [38]:
final_df.to_parquet(OUTPUT_PARQUET_PATH, index=False)

In [39]:
print(OUTPUT_PARQUET_PATH)

/Users/rk8896/DSO585/stock-news-autoencoder/mag7_yf_2021_2025.parquet
